# Chapter 9: Supervised Fine-Tuning (SFT) for MiniMind

## 🎯 Overview

This notebook implements **Supervised Fine-Tuning (SFT)** for your pretrained Transformer model.

**What you'll do:**
1. ✅ Download Alpaca dataset (52K instruction-response pairs)
2. ✅ Format data for training
3. ✅ Load your pretrained model from Chapter 8
4. ✅ Fine-tune on instructions
5. ✅ Test your instruction-following model

**Time required:** 4-6 hours on A100, 16-20 hours on T4

**Prerequisites:**
- ✅ Pretrained model from Chapter 8 (`best_model.pt`)
- ✅ Tokenizer (`tokenizer.json`)
- ✅ GPU runtime (T4 or A100)

---

## 📊 What Changes

| Aspect | Pretraining (Ch 8) | SFT (Ch 9) |
|--------|-------------------|------------|
| Task | Predict next token | Follow instructions |
| Data | Raw text (2.6GB) | 52K instruction pairs |
| Time | 16-24 hours | 4-6 hours |
| Learning rate | 3e-4 | 2e-5 (15x smaller) |
| Output | Text completer | Instruction follower |

# 1. Setup

## Check GPU and Install Dependencies

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import torch
print(f"\n✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

NVIDIA A100-SXM4-40GB, 40960 MiB

✅ PyTorch version: 2.8.0+cu126
✅ CUDA available: True
✅ GPU: NVIDIA A100-SXM4-40GB
✅ Memory: 39.6 GB


**I would strongly suggest that in thsi Chapter you use A100 GPU or at least L4 **

In [ ]:
# Install required packages
!pip install -q tokenizers requests tqdm

print("✅ Dependencies installed!")

✅ Dependencies installed!


## Upload Your Files

**Upload these files from Chapter 8:**
1. `model.py` - Your Transformer model definition
2. `best_model.pt` - Pretrained checkpoint
3. `tokenizer.json` - Tokenizer

Click the folder icon on the left → Upload files

In [ ]:
# Verify uploaded files
import os

required_files = ['model.py', 'best_model.pt', 'tokenizer.json']
missing = [f for f in required_files if not os.path.exists(f)]

if missing:
    print("❌ Missing files:")
    for f in missing:
        print(f"   - {f}")
    print("\nPlease upload them using the file browser on the left.")
else:
    print("✅ All required files found!")
    !ls -lh model.py best_model.pt tokenizer.json

❌ Missing files:
   - model.py
   - best_model.pt
   - tokenizer.json

Please upload them using the file browser on the left.


I saved the above file in Chapter 8 in my Google Drive, so I just mount my gdrive instead.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
cd /content/drive/MyDrive/gpt3/

/content/drive/MyDrive/gpt3


In [ ]:
pwd

'/content/drive/MyDrive/gpt3'

# 3. Download and Prepare Data

## Download Alpaca Dataset (52K Instructions)

In [ ]:
%%writefile download_sft_data.py
import json
import requests
import os
from pathlib import Path

def download_alpaca():
    """Download Stanford Alpaca dataset"""
    print("="*70)
    print("DOWNLOADING ALPACA DATASET")
    print("="*70)

    url = "https://raw.githubusercontent.com/tatsu-lab/stanford_alpaca/main/alpaca_data.json"

    os.makedirs('data/sft', exist_ok=True)
    output_path = 'data/sft/alpaca_data.json'

    print(f"\nDownloading from: {url}")

    response = requests.get(url)
    response.raise_for_status()

    data = response.json()

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Downloaded {len(data):,} instruction examples")

    # Statistics
    with_input = sum(1 for x in data if x.get('input'))

    print("\n" + "="*70)
    print("DATASET STATISTICS")
    print("="*70)
    print(f"  Total examples: {len(data):,}")
    print(f"  With context: {with_input:,}")
    print(f"  Without context: {len(data) - with_input:,}")

    # Show samples
    print("\n" + "="*70)
    print("SAMPLE INSTRUCTIONS")
    print("="*70)
    for i, ex in enumerate(data[:3], 1):
        print(f"\n{i}. Instruction: {ex['instruction'][:80]}...")
        if ex.get('input'):
            print(f"   Input: {ex['input'][:60]}...")
        print(f"   Output: {ex['output'][:80]}...")

    print("="*70)

    return data, output_path

# Download
data, json_path = download_alpaca()

Writing download_sft_data.py


In [ ]:
!python download_sft_data.py

DOWNLOADING ALPACA DATASET


✅ Downloaded 52,002 instruction examples

DATASET STATISTICS
  Total examples: 52,002
  With context: 20,679
  Without context: 31,323

SAMPLE INSTRUCTIONS

1. Instruction: Give three tips for staying healthy....
   Output: 1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. ...

2. Instruction: What are the three primary colors?...
   Output: The three primary colors are red, blue, and yellow....

3. Instruction: Describe the structure of an atom....
   Output: An atom is made up of a nucleus, which contains protons and neutrons, surrounded...


In [ ]:
%%writefile download_finetune_datasets.py
"""
download_finetune_datasets.py - Download Popular Instruction/Conversation Datasets

Downloads and prepares various high-quality instruction finetuning datasets:
1. Stanford Alpaca (52K instructions)
2. ShareGPT (Vicuna format, 90K conversations)
3. OpenAssistant (OASST1, 88K messages)
4. WizardLM (Evol-Instruct)
5. Dolly-15K (Databricks)

Usage:
    python download_finetune_datasets.py --dataset alpaca
    python download_finetune_datasets.py --dataset sharegpt --max-samples 10000
    python download_finetune_datasets.py --dataset all
"""

import json
import requests
import os
from pathlib import Path
import argparse
from tqdm import tqdm

# ============================================================================
# Dataset Downloaders
# ============================================================================

def download_alpaca(output_dir='data/sft', max_samples=None):
    """
    Download Stanford Alpaca dataset (52K instruction examples)

    Format: {"instruction": "...", "input": "...", "output": "..."}
    """
    print("\n" + "="*70)
    print("DOWNLOADING STANFORD ALPACA")
    print("="*70)

    url = "https://raw.githubusercontent.com/tatsu-lab/stanford_alpaca/main/alpaca_data.json"
    output_path = os.path.join(output_dir, 'alpaca_data.json')

    print(f"\nURL: {url}")
    print(f"Output: {output_path}")

    response = requests.get(url)
    response.raise_for_status()

    data = response.json()

    if max_samples:
        data = data[:max_samples]
        print(f"  Limited to {max_samples:,} samples")

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Downloaded {len(data):,} examples")

    # Statistics
    with_input = sum(1 for x in data if x.get('input'))
    print(f"  With context: {with_input:,}")
    print(f"  Without context: {len(data) - with_input:,}")

    # Show sample
    print("\n📝 Sample:")
    print(f"  Instruction: {data[0]['instruction'][:80]}...")
    print(f"  Output: {data[0]['output'][:80]}...")

    return output_path


def download_sharegpt_huggingface(output_dir='data/sft', max_samples=None):
    """
    Download ShareGPT from HuggingFace (cleaned version)

    Format: {"conversations": [{"from": "human/gpt", "value": "..."}]}

    Note: Requires 'datasets' library: pip install datasets
    """
    print("\n" + "="*70)
    print("DOWNLOADING SHAREGPT (HuggingFace)")
    print("="*70)

    try:
        from datasets import load_dataset
    except ImportError:
        print("❌ Error: 'datasets' library not installed")
        print("   Install it with: pip install datasets")
        return None

    print("\nLoading dataset from HuggingFace...")
    print("  Dataset: anon8231489123/ShareGPT_Vicuna_unfiltered")

    # Load dataset
    dataset = load_dataset("anon8231489123/ShareGPT_Vicuna_unfiltered", split="train")

    if max_samples:
        dataset = dataset.select(range(min(max_samples, len(dataset))))
        print(f"  Limited to {max_samples:,} samples")

    print(f"\n✅ Loaded {len(dataset):,} conversations")

    # Convert to JSON
    output_path = os.path.join(output_dir, 'sharegpt_vicuna.json')

    data = []
    for item in tqdm(dataset, desc="Converting"):
        data.append({
            'id': item.get('id', ''),
            'conversations': item['conversations']
        })

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Saved to {output_path}")

    # Statistics
    total_turns = sum(len(item['conversations']) for item in data)
    print(f"  Total conversation turns: {total_turns:,}")
    print(f"  Avg turns per conversation: {total_turns/len(data):.1f}")

    # Show sample
    print("\n📝 Sample conversation:")
    sample = data[0]['conversations']
    for turn in sample[:2]:
        role = turn['from']
        text = turn['value'][:80]
        print(f"  {role}: {text}...")

    return output_path


def download_oasst1(output_dir='data/sft', max_samples=None):
    """
    Download OpenAssistant Conversations (OASST1)

    High-quality human-generated conversations in multiple languages.
    Format: Tree structure with parent/child messages
    """
    print("\n" + "="*70)
    print("DOWNLOADING OPENASSISTANT (OASST1)")
    print("="*70)

    try:
        from datasets import load_dataset
    except ImportError:
        print("❌ Error: 'datasets' library not installed")
        print("   Install it with: pip install datasets")
        return None

    print("\nLoading dataset from HuggingFace...")
    print("  Dataset: OpenAssistant/oasst1")

    # Load dataset
    dataset = load_dataset("OpenAssistant/oasst1", split="train")

    print(f"\n✅ Loaded {len(dataset):,} messages")

    # Convert to conversation format
    # OASST1 has a tree structure, we'll extract linear conversations
    output_path = os.path.join(output_dir, 'oasst1.json')

    # Group by conversation
    conversations = {}
    for msg in dataset:
        conv_id = msg['message_tree_id']
        if conv_id not in conversations:
            conversations[conv_id] = []
        conversations[conv_id].append(msg)

    # Convert to ShareGPT format
    data = []
    for conv_id, messages in tqdm(list(conversations.items())[:max_samples] if max_samples else conversations.items(),
                                   desc="Converting"):
        # Sort by created date
        messages.sort(key=lambda x: x['created_date'])

        # Build conversation
        conv = {
            'id': conv_id,
            'conversations': []
        }

        for msg in messages:
            role = 'human' if msg['role'] == 'prompter' else 'assistant'
            conv['conversations'].append({
                'from': role,
                'value': msg['text']
            })

        # Only include conversations with at least 2 turns
        if len(conv['conversations']) >= 2:
            data.append(conv)

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Saved {len(data):,} conversations to {output_path}")

    # Statistics
    total_turns = sum(len(item['conversations']) for item in data)
    print(f"  Total conversation turns: {total_turns:,}")
    print(f"  Avg turns per conversation: {total_turns/len(data):.1f}")

    return output_path


def download_wizardlm(output_dir='data/sft', max_samples=None):
    """
    Download WizardLM Evol-Instruct dataset

    Complex instructions evolved from Alpaca using LLM
    """
    print("\n" + "="*70)
    print("DOWNLOADING WIZARDLM (Evol-Instruct)")
    print("="*70)

    try:
        from datasets import load_dataset
    except ImportError:
        print("❌ Error: 'datasets' library not installed")
        print("   Install it with: pip install datasets")
        return None

    print("\nLoading dataset from HuggingFace...")
    print("  Dataset: WizardLM/WizardLM_evol_instruct_V2_196k")

    try:
        # Try V2 first
        dataset = load_dataset("WizardLM/WizardLM_evol_instruct_V2_196k", split="train")
    except:
        # Fallback to older version
        print("  Trying older version: cognitivecomputations/WizardLM_alpaca_evol_instruct_70k_unfiltered")
        dataset = load_dataset("cognitivecomputations/WizardLM_alpaca_evol_instruct_70k_unfiltered", split="train")

    if max_samples:
        dataset = dataset.select(range(min(max_samples, len(dataset))))

    print(f"\n✅ Loaded {len(dataset):,} examples")

    # Convert to standard format
    output_path = os.path.join(output_dir, 'wizardlm.json')

    data = []
    for item in tqdm(dataset, desc="Converting"):
        # WizardLM format varies, adapt as needed
        if 'conversations' in item:
            # Format 1: Already in conversation format
            data.append(item)
        elif 'instruction' in item and 'output' in item:
            # Format 2: Alpaca-style
            data.append({
                'instruction': item['instruction'],
                'input': item.get('input', ''),
                'output': item['output']
            })

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Saved to {output_path}")

    return output_path


def download_dolly(output_dir='data/sft', max_samples=None):
    """
    Download Databricks Dolly 15K

    High-quality human-generated instruction-response pairs
    """
    print("\n" + "="*70)
    print("DOWNLOADING DATABRICKS DOLLY-15K")
    print("="*70)

    try:
        from datasets import load_dataset
    except ImportError:
        print("❌ Error: 'datasets' library not installed")
        print("   Install it with: pip install datasets")
        return None

    print("\nLoading dataset from HuggingFace...")
    print("  Dataset: databricks/databricks-dolly-15k")

    dataset = load_dataset("databricks/databricks-dolly-15k", split="train")

    if max_samples:
        dataset = dataset.select(range(min(max_samples, len(dataset))))

    print(f"\n✅ Loaded {len(dataset):,} examples")

    # Convert to standard format
    output_path = os.path.join(output_dir, 'dolly_15k.json')

    data = []
    for item in tqdm(dataset, desc="Converting"):
        data.append({
            'instruction': item['instruction'],
            'input': item.get('context', ''),
            'output': item['response']
        })

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Saved to {output_path}")

    # Statistics
    with_context = sum(1 for x in data if x['input'])
    print(f"  With context: {with_context:,}")
    print(f"  Without context: {len(data) - with_context:,}")

    return output_path


# ============================================================================
# Direct ShareGPT Download (without HuggingFace)
# ============================================================================

def download_sharegpt_direct(output_dir='data/sft', max_samples=None):
    """
    Download ShareGPT dataset directly from GitHub

    This is a smaller, curated version but doesn't require HuggingFace datasets library.
    """
    print("\n" + "="*70)
    print("DOWNLOADING SHAREGPT (Direct from GitHub)")
    print("="*70)

    # Try multiple sources
    urls = [
        "https://huggingface.co/datasets/anon8231489123/ShareGPT_Vicuna_unfiltered/resolve/main/ShareGPT_V3_unfiltered_cleaned_split.json",
        "https://raw.githubusercontent.com/lm-sys/FastChat/main/data/dummy_conversation.json",
    ]

    output_path = os.path.join(output_dir, 'sharegpt_direct.json')

    for url in urls:
        try:
            print(f"\nTrying: {url}")
            response = requests.get(url, timeout=30)
            response.raise_for_status()

            data = response.json()

            if max_samples:
                data = data[:max_samples]

            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2, ensure_ascii=False)

            print(f"\n✅ Downloaded {len(data):,} conversations")
            print(f"   Saved to: {output_path}")

            return output_path

        except Exception as e:
            print(f"   ❌ Failed: {e}")
            continue

    print("\n⚠️  All direct download attempts failed")
    print("   Try using the HuggingFace method instead (requires 'datasets' library)")
    return None


# ============================================================================
# Main Function
# ============================================================================

def main():
    parser = argparse.ArgumentParser(
        description="Download instruction finetuning datasets"
    )

    parser.add_argument(
        "--dataset", "-d",
        type=str,
        default="alpaca",
        choices=["alpaca", "sharegpt", "sharegpt-direct", "oasst1", "wizardlm", "dolly", "all"],
        help="Dataset to download"
    )

    parser.add_argument(
        "--output-dir", "-o",
        type=str,
        default="data/sft",
        help="Output directory"
    )

    parser.add_argument(
        "--max-samples", "-n",
        type=int,
        default=None,
        help="Maximum number of samples to download"
    )

    args = parser.parse_args()

    # Create output directory
    os.makedirs(args.output_dir, exist_ok=True)

    print("="*70)
    print("INSTRUCTION DATASET DOWNLOADER")
    print("="*70)
    print(f"\nOutput directory: {args.output_dir}")
    if args.max_samples:
        print(f"Max samples: {args.max_samples:,}")
    print()

    # Download datasets
    downloaded = []

    if args.dataset == "all":
        datasets_to_download = ["alpaca", "sharegpt", "oasst1", "dolly"]
    else:
        datasets_to_download = [args.dataset]

    for dataset_name in datasets_to_download:
        try:
            if dataset_name == "alpaca":
                path = download_alpaca(args.output_dir, args.max_samples)
            elif dataset_name == "sharegpt":
                path = download_sharegpt_huggingface(args.output_dir, args.max_samples)
            elif dataset_name == "sharegpt-direct":
                path = download_sharegpt_direct(args.output_dir, args.max_samples)
            elif dataset_name == "oasst1":
                path = download_oasst1(args.output_dir, args.max_samples)
            elif dataset_name == "wizardlm":
                path = download_wizardlm(args.output_dir, args.max_samples)
            elif dataset_name == "dolly":
                path = download_dolly(args.output_dir, args.max_samples)

            if path:
                downloaded.append((dataset_name, path))

        except Exception as e:
            print(f"\n❌ Error downloading {dataset_name}: {e}")
            continue

    # Summary
    print("\n" + "="*70)
    print("DOWNLOAD COMPLETE")
    print("="*70)

    if downloaded:
        print("\n✅ Successfully downloaded:")
        for name, path in downloaded:
            size_mb = os.path.getsize(path) / (1024**2)
            print(f"  {name:15s} → {path} ({size_mb:.2f} MB)")

        print("\n📝 Next steps:")
        print("  1. Prepare data for finetuning:")
        for name, path in downloaded:
            output_name = Path(path).stem
            print(f"     python 4_prepare_finetune_data.py -i {path} -o data/processed/finetune_{output_name}.jsonl")

        print("\n  2. Start finetuning:")
        print("     python 5_finetune_ddp.py --data_path data/processed/finetune_XXX.jsonl [other args]")
    else:
        print("\n❌ No datasets were downloaded successfully")
        print("\n💡 Troubleshooting:")
        print("  - Install datasets library: pip install datasets")
        print("  - Check internet connection")
        print("  - Try different dataset: --dataset alpaca")


if __name__ == "__main__":
    main()

Overwriting download_finetune_datasets.py


In [ ]:
!python download_finetune_datasets.py --dataset sharegpt --max-samples 10000

INSTRUCTION DATASET DOWNLOADER

Output directory: data/sft
Max samples: 10,000


DOWNLOADING SHAREGPT (HuggingFace)

Loading dataset from HuggingFace...
  Dataset: anon8231489123/ShareGPT_Vicuna_unfiltered
README.md: 4.41kB [00:00, 19.1MB/s]

❌ Error downloading sharegpt: No (supported) data files found in anon8231489123/ShareGPT_Vicuna_unfiltered

DOWNLOAD COMPLETE

❌ No datasets were downloaded successfully

💡 Troubleshooting:
  - Install datasets library: pip install datasets
  - Check internet connection
  - Try different dataset: --dataset alpaca


# 4. Dataset and DataLoader and Convert to JSONL

In [ ]:
%%writefile prepare_sft_data.py
"""
prepare_finetune_data.py - Prepare Instruction Finetuning Dataset

Converts various instruction formats to a standardized format for finetuning.

1) python prepare_sft_data.py --input "/content/drive/MyDrive/gpt3/data/sft/alpaca_data.json"
2) python 8_prepare_sft_data.py --input "data/sft/alpaca_data.json"


Supported formats:
1. Alpaca format: {"instruction": "...", "input": "...", "output": "..."}
2. ShareGPT format: {"conversations": [{"from": "human/gpt", "value": "..."}]}
3. Simple format: {"prompt": "...", "response": "..."}

Output format: JSONL with {"instruction": "...", "response": "..."}
"""

import os
import json
import argparse
from pathlib import Path
from tqdm import tqdm

print("="*70)
print("FINETUNING DATA PREPARATION")
print("="*70)
print("\nConverts various instruction formats to standardized format")
print("="*70 + "\n")

# ============================================================================
# Format Converters
# ============================================================================

def convert_alpaca_format(item):
    """
    Convert Alpaca format to standard format.

    Alpaca format:
    {
        "instruction": "Write a poem about...",
        "input": "",  # optional context
        "output": "Roses are red..."
    }
    """
    instruction = item.get('instruction', '').strip()
    input_text = item.get('input', '').strip()
    output = item.get('output', '').strip()

    if not instruction or not output:
        return None

    # Combine instruction and input
    if input_text:
        full_instruction = f"{instruction}\n\n{input_text}"
    else:
        full_instruction = instruction

    return {
        'instruction': full_instruction,
        'response': output
    }

def convert_sharegpt_format(item):
    """
    Convert ShareGPT format to standard format.

    ShareGPT format:
    {
        "conversations": [
            {"from": "human", "value": "Hello!"},
            {"from": "gpt", "value": "Hi there!"},
            {"from": "human", "value": "How are you?"},
            {"from": "gpt", "value": "I'm doing well!"}
        ]
    }

    Returns multiple instruction-response pairs from multi-turn conversations.
    """
    conversations = item.get('conversations', [])

    if not conversations:
        return None

    pairs = []
    context = []

    for i, turn in enumerate(conversations):
        role = turn.get('from', '').lower()
        content = turn.get('value', '').strip()

        if not content:
            continue

        if role in ['human', 'user']:
            # Build instruction with context from previous turns
            if context:
                full_instruction = '\n\n'.join(context) + '\n\n' + content
            else:
                full_instruction = content

            # Look ahead for the response
            if i + 1 < len(conversations):
                next_turn = conversations[i + 1]
                if next_turn.get('from', '').lower() in ['gpt', 'assistant']:
                    response = next_turn.get('value', '').strip()
                    if response:
                        pairs.append({
                            'instruction': full_instruction,
                            'response': response
                        })
                        # Add this exchange to context for future turns
                        context.append(f"User: {content}")
                        context.append(f"Assistant: {response}")

    return pairs if pairs else None

def convert_simple_format(item):
    """
    Convert simple prompt-response format.

    Simple format:
    {
        "prompt": "What is the capital of France?",
        "response": "The capital of France is Paris."
    }
    """
    instruction = item.get('prompt', '').strip() or item.get('question', '').strip()
    response = item.get('response', '').strip() or item.get('answer', '').strip()

    if not instruction or not response:
        return None

    return {
        'instruction': instruction,
        'response': response
    }

def detect_format(item):
    """Detect the format of the input data"""
    if 'conversations' in item:
        return 'sharegpt'
    elif 'instruction' in item and 'output' in item:
        return 'alpaca'
    elif 'prompt' in item or 'question' in item:
        return 'simple'
    else:
        return 'unknown'

def convert_item(item):
    """Convert an item to standard format based on detected format"""
    format_type = detect_format(item)

    if format_type == 'alpaca':
        return convert_alpaca_format(item)
    elif format_type == 'sharegpt':
        return convert_sharegpt_format(item)
    elif format_type == 'simple':
        return convert_simple_format(item)
    else:
        return None

# ============================================================================
# Data Quality Filters
# ============================================================================

def filter_item(item, min_instruction_len=10, max_instruction_len=2048,
                min_response_len=10, max_response_len=4096):
    """
    Filter out low-quality samples.

    Criteria:
    - Instruction and response within length limits
    - No empty or trivial content
    - Reasonable length ratio
    """
    instruction = item['instruction']
    response = item['response']

    # Length checks
    if len(instruction) < min_instruction_len or len(instruction) > max_instruction_len:
        return False

    if len(response) < min_response_len or len(response) > max_response_len:
        return False

    # Check for trivial responses
    trivial_responses = [
        'ok', 'yes', 'no', 'sure', 'thanks', 'thank you', 'okay'
    ]
    if response.lower().strip() in trivial_responses:
        return False

    # Check length ratio (response should be substantial)
    if len(response) < len(instruction) * 0.3:
        return False

    return True

# ============================================================================
# Main Processing Function
# ============================================================================

def process_dataset(input_path, output_path, format_type='auto',
                   max_samples=None, apply_filters=True):
    """
    Process input dataset and convert to standard format.

    Args:
        input_path: Path to input JSONL file
        output_path: Path to output JSONL file
        format_type: Input format ('auto', 'alpaca', 'sharegpt', 'simple')
        max_samples: Maximum number of samples to process (None = all)
        apply_filters: Whether to apply quality filters
    """
    print(f"📂 Input file: {input_path}")
    print(f"💾 Output file: {output_path}")
    print(f"📋 Format: {format_type}")
    print()

    # Read input data
    print("Loading data...")
    input_data = []

    input_ext = Path(input_path).suffix.lower()

    if input_ext == '.jsonl':
        with open(input_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    try:
                        input_data.append(json.loads(line))
                    except json.JSONDecodeError:
                        continue
    elif input_ext == '.json':
        with open(input_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            if isinstance(data, list):
                input_data = data
            else:
                input_data = [data]
    else:
        raise ValueError(f"Unsupported file format: {input_ext}")

    print(f"✅ Loaded {len(input_data):,} raw samples")

    if max_samples:
        input_data = input_data[:max_samples]
        print(f"   Limited to {max_samples:,} samples")

    # Convert samples
    print("\nConverting to standard format...")
    output_data = []
    format_counts = {'alpaca': 0, 'sharegpt': 0, 'simple': 0, 'unknown': 0}

    for item in tqdm(input_data, desc="Processing"):
        if format_type == 'auto':
            detected_format = detect_format(item)
            format_counts[detected_format] += 1
        else:
            detected_format = format_type

        converted = convert_item(item)

        if converted is None:
            continue

        # Handle ShareGPT which can return multiple pairs
        if isinstance(converted, list):
            output_data.extend(converted)
        else:
            output_data.append(converted)

    print(f"\n✅ Converted {len(output_data):,} instruction-response pairs")

    if format_type == 'auto':
        print(f"\n📊 Format distribution:")
        for fmt, count in format_counts.items():
            if count > 0:
                print(f"   {fmt}: {count:,} samples")

    # Apply filters
    if apply_filters:
        print("\nApplying quality filters...")
        filtered_data = [item for item in output_data if filter_item(item)]
        print(f"✅ Kept {len(filtered_data):,} samples after filtering")
        print(f"   Removed {len(output_data) - len(filtered_data):,} low-quality samples")
        output_data = filtered_data

    # Save output
    print(f"\nSaving to {output_path}...")
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with open(output_path, 'w', encoding='utf-8') as f:
        for item in output_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

    print(f"✅ Saved {len(output_data):,} samples")

    # Show statistics
    print("\n" + "="*70)
    print("STATISTICS")
    print("="*70)

    instruction_lengths = [len(item['instruction']) for item in output_data]
    response_lengths = [len(item['response']) for item in output_data]

    print(f"\nInstruction length:")
    print(f"   Min: {min(instruction_lengths):,} chars")
    print(f"   Max: {max(instruction_lengths):,} chars")
    print(f"   Avg: {sum(instruction_lengths)//len(instruction_lengths):,} chars")

    print(f"\nResponse length:")
    print(f"   Min: {min(response_lengths):,} chars")
    print(f"   Max: {max(response_lengths):,} chars")
    print(f"   Avg: {sum(response_lengths)//len(response_lengths):,} chars")

    # Show sample
    print("\n" + "="*70)
    print("SAMPLE DATA")
    print("="*70)

    sample = output_data[0]
    print(f"\nInstruction: {sample['instruction'][:200]}...")
    print(f"\nResponse: {sample['response'][:200]}...")

    print("\n" + "="*70)
    print("✅ PREPARATION COMPLETE!")
    print("="*70)
    print(f"\n📄 Output file: {output_path}")
    print(f"📊 Total samples: {len(output_data):,}")
    print(f"\n🚀 Ready for finetuning!")

# ============================================================================
# Command Line Interface
# ============================================================================

if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="Prepare instruction finetuning dataset"
    )

    parser.add_argument(
        "--input", "-i",
        type=str,
        required=True,
        help="Input file path (JSON or JSONL)"
    )

    parser.add_argument(
        "--output", "-o",
        type=str,
        default="data/processed/finetune.jsonl",
        help="Output file path (JSONL)"
    )

    parser.add_argument(
        "--format", "-f",
        type=str,
        default="auto",
        choices=['auto', 'alpaca', 'sharegpt', 'simple'],
        help="Input data format (auto-detect by default)"
    )

    parser.add_argument(
        "--max-samples", "-n",
        type=int,
        default=None,
        help="Maximum number of samples to process"
    )

    parser.add_argument(
        "--no-filter",
        action="store_true",
        help="Disable quality filtering"
    )

    args = parser.parse_args()

    process_dataset(
        input_path=args.input,
        output_path=args.output,
        format_type=args.format,
        max_samples=args.max_samples,
        apply_filters=not args.no_filter
    )


Overwriting prepare_sft_data.py


In [ ]:
!python prepare_sft_data.py --input "data/sft/alpaca_data.json"

FINETUNING DATA PREPARATION

Converts various instruction formats to standardized format

📂 Input file: data/sft/alpaca_data.json
💾 Output file: data/processed/finetune.jsonl
📋 Format: auto

Loading data...
✅ Loaded 52,002 raw samples

Converting to standard format...
Processing: 100% 52002/52002 [00:00<00:00, 758806.84it/s]

✅ Converted 51,974 instruction-response pairs

📊 Format distribution:
   alpaca: 52,002 samples

Applying quality filters...
✅ Kept 46,178 samples after filtering
   Removed 5,796 low-quality samples

Saving to data/processed/finetune.jsonl...
✅ Saved 46,178 samples

STATISTICS

Instruction length:
   Min: 14 chars
   Max: 1,564 chars
   Avg: 80 chars

Response length:
   Min: 10 chars
   Max: 3,668 chars
   Avg: 302 chars

SAMPLE DATA

Instruction: Give three tips for staying healthy....

Response: 1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. 
2. Exercise regularly to keep your body active and strong. 
3. Get enough sleep and ma

# 5. Load Pretrained Model

# 6. Training Setup

# 7. Training Loop

In [ ]:
pwd

'/content/drive/MyDrive/gpt3'

In [ ]:
!python finetune_ddp.py --data_path data/processed/finetune.jsonl \
    --tokenizer_path model/tokenizer.json \
    --save_dir out_finetune \
    --pretrained_checkpoint out/checkpoints/checkpoint_latest.pt \
    --hidden_size 512 \
    --num_hidden_layers 8 \
    --num_attention_heads 16 \
    --num_key_value_heads 8 \
    --max_seq_len 512 \
    --dropout 0.0 \
    --use_moe 0 \
    --epochs 5 \
    --batch_size 32 \
    --learning_rate 1e-5 \
    --min_lr 1e-6 \
    --warmup_steps 500 \
    --weight_decay 0.01 \
    --grad_clip 1.0 \
    --accumulation_steps 1 \
    --dtype bfloat16 \
    --num_workers 4 \
    --log_interval 100 \
    --save_interval 500

In [ ]:
%%writefile finetune_ddp.py
"""
finetune_ddp_fixed_checkpoints.py - MiniMind Instruction Finetuning with Proper Checkpoint Saving

FIXES:
- Saves checkpoints at end of each epoch
- Saves numbered checkpoints at save_interval
- Easy to test different epochs

Usage:
    python finetune_ddp_fixed_checkpoints.py [args]
"""

import os
import sys
import argparse
import time
import warnings
import torch
import torch.distributed as dist
from contextlib import nullcontext
from torch import optim, nn
from torch.nn.parallel import DistributedDataParallel
from torch.utils.data import DataLoader, DistributedSampler
from pathlib import Path
import json

warnings.filterwarnings('ignore')

# ============================================================================
# Import MiniMind Model
# ============================================================================
try:
    from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
    USE_MINIMIND_MODEL = True
except ImportError:
    print("⚠️  Could not import MiniMind model")
    USE_MINIMIND_MODEL = False

# ============================================================================
# Tokenizer Wrapper
# ============================================================================
class EnglishTokenizer:
    """Wrapper for English BPE tokenizer with correct special tokens"""
    def __init__(self, tokenizer_path='model/tokenizer.json'):
        from tokenizers import Tokenizer

        print(f"Loading tokenizer from: {tokenizer_path}")
        self.tokenizer = Tokenizer.from_file(tokenizer_path)
        self.vocab_size = self.tokenizer.get_vocab_size()

        vocab = self.tokenizer.get_vocab()

        # Use correct special token names
        self.pad_id = vocab.get('<pad>', 0)
        self.unk_id = vocab.get('<unk>', 1)
        self.bos_id = vocab.get('<s>', 2)
        self.eos_id = vocab.get('</s>', 3)

        # Verify tokens exist
        if self.pad_id == self.bos_id == self.eos_id == 0:
            raise ValueError(
                "❌ ERROR: Special tokens not found! "
                "Please use correct tokenizer with: <pad>, <unk>, <s>, </s>"
            )

        print(f"✅ Tokenizer loaded:")
        print(f"   Vocab size: {self.vocab_size:,}")
        print(f"   PAD: {self.pad_id} (<pad>)")
        print(f"   UNK: {self.unk_id} (<unk>)")
        print(f"   BOS: {self.bos_id} (<s>)")
        print(f"   EOS: {self.eos_id} (</s>)")

    def encode(self, text, add_special_tokens=True):
        """Encode text to token IDs"""
        encoding = self.tokenizer.encode(text)
        ids = encoding.ids
        if add_special_tokens:
            ids = [self.bos_id] + ids + [self.eos_id]
        return ids

    def decode(self, ids):
        """Decode token IDs to text"""
        if isinstance(ids, torch.Tensor):
            ids = ids.tolist()
        # Remove special tokens
        ids = [id for id in ids if id not in [self.pad_id, self.bos_id, self.eos_id]]
        return self.tokenizer.decode(ids)

    def __len__(self):
        return self.vocab_size

# ============================================================================
# Finetuning Dataset
# ============================================================================
class FinetuneDataset(torch.utils.data.Dataset):
    """
    Instruction finetuning dataset with proper loss masking.

    Format: {"instruction": "...", "response": "..."}

    Creates training examples where:
    - Input: <s> {instruction} {response} </s>
    - Loss is only computed on response tokens (not instruction)
    """
    def __init__(self, data_path, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.max_length = max_length

        print(f"Loading finetuning data from {data_path}...")
        self.samples = []

        with open(data_path, 'r', encoding='utf-8') as f:
            for line in f:
                data = json.loads(line.strip())
                if data.get('instruction') and data.get('response'):
                    self.samples.append(data)

        print(f"✅ Loaded {len(self.samples):,} instruction-response pairs")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        instruction = sample['instruction']
        response = sample['response']

        # Encode instruction and response separately
        instruction_tokens = self.tokenizer.encode(instruction, add_special_tokens=False)
        response_tokens = self.tokenizer.encode(response, add_special_tokens=False)

        # Construct full sequence: <s> instruction response </s>
        tokens = [self.tokenizer.bos_id] + instruction_tokens + response_tokens + [self.tokenizer.eos_id]

        # Calculate where response starts
        response_start = 1 + len(instruction_tokens)

        # Truncate if too long
        if len(tokens) > self.max_length + 1:
            overflow = len(tokens) - (self.max_length + 1)
            if overflow < len(instruction_tokens):
                instruction_tokens = instruction_tokens[overflow:]
                tokens = [self.tokenizer.bos_id] + instruction_tokens + response_tokens + [self.tokenizer.eos_id]
                response_start = 1 + len(instruction_tokens)
            else:
                tokens = tokens[:self.max_length + 1]

        # Convert to tensor
        tokens_tensor = torch.tensor(tokens, dtype=torch.long)

        # Pad if needed
        if len(tokens_tensor) < self.max_length + 1:
            padding = torch.full(
                (self.max_length + 1 - len(tokens_tensor),),
                self.tokenizer.pad_id,
                dtype=torch.long
            )
            tokens_tensor = torch.cat([tokens_tensor, padding])

        # Create input and target
        x = tokens_tensor[:-1]
        y = tokens_tensor[1:]

        # Create loss mask - only compute loss on response tokens
        loss_mask = torch.zeros_like(y, dtype=torch.float)

        response_mask_start = max(0, response_start - 1)
        response_mask_end = min(len(loss_mask), response_start - 1 + len(response_tokens) + 1)
        loss_mask[response_mask_start:response_mask_end] = 1.0

        # Disable loss on padding
        loss_mask[y == self.tokenizer.pad_id] = 0.0

        return x, y, loss_mask

# ============================================================================
# Training Utilities
# ============================================================================

def init_distributed_mode():
    """Initialize distributed training"""
    if 'RANK' in os.environ and 'WORLD_SIZE' in os.environ:
        rank = int(os.environ["RANK"])
        world_size = int(os.environ['WORLD_SIZE'])
        local_rank = int(os.environ['LOCAL_RANK'])
    else:
        print('Not using distributed mode')
        return None

    torch.cuda.set_device(local_rank)
    dist.init_process_group(
        backend='nccl',
        init_method='env://',
        world_size=world_size,
        rank=rank
    )
    dist.barrier()
    return local_rank

def is_main_process():
    """Check if current process is main"""
    return not dist.is_initialized() or dist.get_rank() == 0

def setup_seed(seed):
    """Set random seed for reproducibility"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def get_lr(step, total_steps, learning_rate, warmup_steps=500, min_lr=None):
    """Learning rate schedule: warmup + cosine decay"""
    if min_lr is None:
        min_lr = learning_rate * 0.1

    # Warmup
    if step < warmup_steps:
        return learning_rate * step / warmup_steps

    # Cosine decay
    if step > total_steps:
        return min_lr

    decay_ratio = (step - warmup_steps) / (total_steps - warmup_steps)
    coeff = 0.5 * (1.0 + torch.cos(torch.tensor(decay_ratio * 3.14159)))
    return min_lr + coeff * (learning_rate - min_lr)

def Logger(msg):
    """Print log message with timestamp"""
    if is_main_process():
        print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

def save_checkpoint(model, optimizer, scaler, epoch, step, save_dir, is_best=False, save_numbered=False):
    """
    Save training checkpoint

    Args:
        save_numbered: If True, saves checkpoint_epoch_X_step_Y.pt
    """
    if not is_main_process():
        return

    checkpoint_dir = os.path.join(save_dir, 'checkpoints')
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Get model state dict (handle DDP wrapper)
    model_to_save = model.module if hasattr(model, 'module') else model

    checkpoint = {
        'epoch': epoch,
        'step': step,
        'model_state_dict': model_to_save.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict() if scaler.is_enabled() else None,
    }

    # Save latest checkpoint (always)
    checkpoint_path = os.path.join(checkpoint_dir, 'checkpoint_latest.pt')
    torch.save(checkpoint, checkpoint_path)
    Logger(f"💾 Saved checkpoint: {checkpoint_path}")

    # Save numbered checkpoint (at regular intervals or end of epoch)
    if save_numbered:
        checkpoint_path = os.path.join(checkpoint_dir, f'checkpoint_epoch_{epoch}_step_{step}.pt')
        torch.save(checkpoint, checkpoint_path)
        Logger(f"💾 Saved numbered checkpoint: {checkpoint_path}")

    # Save best checkpoint (when loss improves)
    if is_best:
        checkpoint_path = os.path.join(checkpoint_dir, 'checkpoint_best.pt')
        torch.save(checkpoint, checkpoint_path)
        Logger(f"💾 Saved best checkpoint: {checkpoint_path}")

def load_checkpoint(checkpoint_path, model, optimizer, scaler):
    """Load training checkpoint"""
    if not os.path.exists(checkpoint_path):
        Logger(f"⚠️  Checkpoint not found: {checkpoint_path}")
        return None

    Logger(f"Loading checkpoint from {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location='cpu')

    # Load model state
    model_to_load = model.module if hasattr(model, 'module') else model
    model_to_load.load_state_dict(checkpoint['model_state_dict'])

    # Load optimizer state
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    # Load scaler state if available
    if checkpoint.get('scaler_state_dict') and scaler.is_enabled():
        scaler.load_state_dict(checkpoint['scaler_state_dict'])

    Logger(f"✅ Loaded checkpoint from epoch {checkpoint['epoch']}, step {checkpoint.get('step', 0)}")
    return checkpoint

# ============================================================================
# Training Loop
# ============================================================================

def train_epoch(epoch, loader, iters, model, optimizer, scaler, args, lm_config,
                autocast_ctx, start_step=0, wandb=None):
    """Train for one epoch"""
    model.train()

    total_loss = 0
    total_acc = 0
    total_steps = args.epochs * iters
    log_loss = 0
    log_acc = 0
    best_loss = float('inf')

    for step, (x, y, loss_mask) in enumerate(loader, start=start_step):
        actual_step = epoch * iters + step

        # Update learning rate
        lr = get_lr(
            actual_step,
            total_steps,
            args.learning_rate,
            args.warmup_steps,
            args.min_lr
        )
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        # Move data to device
        x = x.to(args.device)
        y = y.to(args.device)
        loss_mask = loss_mask.to(args.device)

        # Forward pass with mixed precision
        with autocast_ctx:
            out = model(x, logits_to_keep=0)
            logits = out.logits

            # Reshape for loss computation
            logits = logits.reshape(-1, logits.size(-1))
            y_flat = y.reshape(-1)
            loss_mask_flat = loss_mask.reshape(-1)

            # Compute loss (only on response tokens)
            loss = nn.functional.cross_entropy(
                logits, y_flat, reduction='none'
            )
            loss = (loss * loss_mask_flat).sum() / loss_mask_flat.sum()

            # Add auxiliary loss if using MoE
            if hasattr(out, 'aux_loss') and out.aux_loss is not None:
                loss = loss + lm_config.aux_loss_alpha * out.aux_loss

            # Scale loss for gradient accumulation
            loss = loss / args.accumulation_steps

        # Backward pass
        scaler.scale(loss).backward()

        # Compute accuracy (only on response tokens)
        with torch.no_grad():
            preds = torch.argmax(logits, dim=-1)
            correct = (preds == y_flat) * loss_mask_flat
            acc = correct.sum() / loss_mask_flat.sum()

        # Update metrics
        total_loss += loss.item() * args.accumulation_steps
        total_acc += acc.item()
        log_loss += loss.item() * args.accumulation_steps
        log_acc += acc.item()

        # Gradient accumulation
        if (step + 1) % args.accumulation_steps == 0:
            # Gradient clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)

            # Optimizer step
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        # Logging
        if (step + 1) % args.log_interval == 0:
            avg_loss = log_loss / args.log_interval
            avg_acc = log_acc / args.log_interval

            Logger(
                f"Epoch {epoch+1}/{args.epochs} | "
                f"Step {step+1}/{iters} | "
                f"Loss: {avg_loss:.4f} | "
                f"Acc: {avg_acc:.4f} | "
                f"LR: {lr:.6f}"
            )

            if wandb:
                wandb.log({
                    'train/loss': avg_loss,
                    'train/accuracy': avg_acc,
                    'train/learning_rate': lr,
                    'train/epoch': epoch,
                    'train/step': actual_step
                })

            log_loss = 0
            log_acc = 0

        # Checkpointing during epoch (at save_interval)
        if (step + 1) % args.save_interval == 0:
            is_best = (avg_loss < best_loss)
            if is_best:
                best_loss = avg_loss
            # Save numbered checkpoint at regular intervals
            save_checkpoint(model, optimizer, scaler, epoch, step + 1,
                          args.save_dir, is_best=is_best, save_numbered=True)

    avg_epoch_loss = total_loss / iters
    avg_epoch_acc = total_acc / iters

    Logger(
        f"Epoch {epoch+1} complete | "
        f"Avg Loss: {avg_epoch_loss:.4f} | "
        f"Avg Acc: {avg_epoch_acc:.4f}"
    )

    # ⭐ IMPORTANT: Save checkpoint at end of each epoch
    is_best = (avg_epoch_loss < best_loss)
    save_checkpoint(model, optimizer, scaler, epoch, iters,
                   args.save_dir, is_best=is_best, save_numbered=True)

# ============================================================================
# Main Training Function
# ============================================================================

def main(args):
    # ========== 1. Initialize distributed training ==========
    local_rank = init_distributed_mode()
    setup_seed(42)

    if local_rank is not None:
        args.device = f'cuda:{local_rank}'
        Logger(f"Initialized DDP with rank {dist.get_rank()}/{dist.get_world_size()}")
    else:
        args.device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
        Logger("Running in single GPU mode")

    # ========== 2. Load tokenizer ==========
    Logger("Loading tokenizer...")
    tokenizer = EnglishTokenizer(args.tokenizer_path)

    # ========== 3. Create model config ==========
    Logger("Creating model configuration...")
    lm_config = MiniMindConfig(
        hidden_size=args.hidden_size,
        num_hidden_layers=args.num_hidden_layers,
        num_attention_heads=args.num_attention_heads,
        num_key_value_heads=args.num_key_value_heads,
        vocab_size=len(tokenizer),
        max_position_embeddings=args.max_seq_len,
        dropout=args.dropout,
        use_moe=bool(args.use_moe),
        bos_token_id=tokenizer.bos_id,
        eos_token_id=tokenizer.eos_id,
    )

    Logger("Model configuration:")
    Logger(f"  Hidden size: {lm_config.hidden_size}")
    Logger(f"  Layers: {lm_config.num_hidden_layers}")
    Logger(f"  Attention heads: {lm_config.num_attention_heads}")
    Logger(f"  KV heads: {lm_config.num_key_value_heads}")
    Logger(f"  Vocab size: {lm_config.vocab_size}")
    Logger(f"  Max seq length: {args.max_seq_len}")
    Logger(f"  Use MoE: {lm_config.use_moe}")

    # ========== 4. Create model ==========
    Logger("Creating model...")

    if USE_MINIMIND_MODEL:
        model = MiniMindForCausalLM(lm_config)
    else:
        raise ImportError(
            "MiniMind model not found. Please ensure model/model_minimind.py exists"
        )

    # Load pretrained checkpoint if specified
    if args.pretrained_checkpoint:
        Logger(f"Loading pretrained weights from {args.pretrained_checkpoint}...")
        checkpoint = torch.load(args.pretrained_checkpoint, map_location='cpu')

        # Handle different checkpoint formats
        if 'model_state_dict' in checkpoint:
            state_dict = checkpoint['model_state_dict']
            Logger("  Using 'model_state_dict' key")
        elif 'model' in checkpoint:
            state_dict = checkpoint['model']
            Logger("  Using 'model' key from pretraining checkpoint")
        else:
            state_dict = checkpoint
            Logger("  Using checkpoint directly as state dict")

        # Load state dict
        missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)

        if missing_keys:
            Logger(f"⚠️  Missing keys ({len(missing_keys)} total): {missing_keys[:3]}")
        if unexpected_keys:
            Logger(f"⚠️  Unexpected keys ({len(unexpected_keys)} total): {unexpected_keys[:3]}")

        if not missing_keys:
            Logger("✅ Loaded pretrained weights successfully!")

    model = model.to(args.device)

    n_params = sum(p.numel() for p in model.parameters())
    Logger(f"Model parameters: {n_params/1e6:.2f}M")

    # ========== 5. Setup mixed precision ==========
    device_type = "cuda" if "cuda" in args.device else "cpu"
    dtype = torch.bfloat16 if args.dtype == "bfloat16" else torch.float16
    autocast_ctx = (
        nullcontext() if device_type == "cpu"
        else torch.cuda.amp.autocast(dtype=dtype)
    )
    scaler = torch.cuda.amp.GradScaler(enabled=(args.dtype == 'float16'))

    # ========== 6. Create dataset and dataloader ==========
    Logger("Creating dataset...")
    train_ds = FinetuneDataset(args.data_path, tokenizer, args.max_seq_len)

    train_sampler = None
    if dist.is_initialized():
        train_sampler = DistributedSampler(
            train_ds,
            num_replicas=dist.get_world_size(),
            rank=dist.get_rank(),
            shuffle=True
        )

    # ========== 7. Create optimizer ==========
    optimizer = optim.AdamW(
        model.parameters(),
        lr=args.learning_rate,
        weight_decay=args.weight_decay,
        betas=(0.9, 0.95)
    )

    # ========== 8. Load checkpoint if resuming ==========
    start_epoch = 0
    start_step = 0

    if args.resume:
        checkpoint_path = os.path.join(args.save_dir, 'checkpoints', 'checkpoint_latest.pt')
        checkpoint = load_checkpoint(checkpoint_path, model, optimizer, scaler)
        if checkpoint:
            start_epoch = checkpoint['epoch']
            start_step = checkpoint.get('step', 0)

    # ========== 9. Wrap model with DDP ==========
    if dist.is_initialized():
        Logger("Wrapping model with DistributedDataParallel...")
        if hasattr(model, 'model'):
            model.model._ddp_params_and_buffers_to_ignore = {"freqs_cos", "freqs_sin"}
        model = DistributedDataParallel(
            model,
            device_ids=[local_rank],
            output_device=local_rank
        )

    # ========== 10. Setup WandB ==========
    wandb = None
    if args.use_wandb and is_main_process():
        try:
            import wandb
            wandb.init(
                project=args.wandb_project,
                name=f"MiniMind-Finetune-{args.hidden_size}",
                config=vars(args)
            )
            Logger("✅ WandB initialized")
        except ImportError:
            Logger("⚠️  WandB not available, skipping")
            wandb = None

    # ========== 11. Start training ==========
    Logger("\n" + "="*70)
    Logger("STARTING FINETUNING")
    Logger("="*70)
    Logger(f"Total epochs: {args.epochs}")
    Logger(f"Batch size per GPU: {args.batch_size}")
    if dist.is_initialized():
        Logger(f"Global batch size: {args.batch_size * dist.get_world_size()}")
    Logger(f"Gradient accumulation steps: {args.accumulation_steps}")
    Logger(f"Effective batch size: {args.batch_size * args.accumulation_steps}")
    Logger(f"Save interval: {args.save_interval} steps")
    Logger("="*70 + "\n")

    for epoch in range(start_epoch, args.epochs):
        if train_sampler:
            train_sampler.set_epoch(epoch)

        # Normal training
        loader = DataLoader(
            train_ds,
            batch_size=args.batch_size,
            shuffle=(train_sampler is None),
            sampler=train_sampler,
            num_workers=args.num_workers,
            pin_memory=True,
            drop_last=True
        )
        iters = len(loader)
        train_epoch(
            epoch, loader, iters, model, optimizer, scaler,
            args, lm_config, autocast_ctx, 0, wandb
        )

    Logger("\n" + "="*70)
    Logger("🎉 FINETUNING COMPLETE!")
    Logger("="*70)

    # Save final model
    if is_main_process():
        final_model_dir = os.path.join(args.save_dir, 'final_model')
        os.makedirs(final_model_dir, exist_ok=True)

        model_to_save = model.module if hasattr(model, 'module') else model

        try:
            model_to_save.save_pretrained(final_model_dir, safe_serialization=False)
            Logger(f"✅ Saved final model to {final_model_dir} (using PyTorch format)")
        except Exception as e:
            Logger(f"⚠️  save_pretrained failed: {e}")
            Logger(f"💾 Saving model state dict directly instead...")

            torch.save({
                'model_state_dict': model_to_save.state_dict(),
                'config': model_to_save.config,
            }, os.path.join(final_model_dir, 'pytorch_model.bin'))

            model_to_save.config.save_pretrained(final_model_dir)
            Logger(f"✅ Saved model state dict to {final_model_dir}")

    if wandb:
        wandb.finish()

    if dist.is_initialized():
        dist.destroy_process_group()

# ============================================================================
# Entry Point
# ============================================================================

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="MiniMind Instruction Finetuning")

    # Paths
    parser.add_argument("--data_path", type=str, default="data/processed/finetune.jsonl")
    parser.add_argument("--tokenizer_path", type=str, default="model/tokenizer.json")
    parser.add_argument("--save_dir", type=str, default="out_finetune")
    parser.add_argument("--pretrained_checkpoint", type=str, default=None)

    # Model architecture
    parser.add_argument("--hidden_size", type=int, default=768)
    parser.add_argument("--num_hidden_layers", type=int, default=16)
    parser.add_argument("--num_attention_heads", type=int, default=16)
    parser.add_argument("--num_key_value_heads", type=int, default=8)
    parser.add_argument("--max_seq_len", type=int, default=512)
    parser.add_argument("--dropout", type=float, default=0.0)
    parser.add_argument("--use_moe", type=int, default=0, choices=[0, 1])

    # Training hyperparameters
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--learning_rate", type=float, default=1e-5)
    parser.add_argument("--min_lr", type=float, default=1e-6)
    parser.add_argument("--warmup_steps", type=int, default=500)
    parser.add_argument("--weight_decay", type=float, default=0.01)
    parser.add_argument("--grad_clip", type=float, default=1.0)
    parser.add_argument("--accumulation_steps", type=int, default=1)

    # Training configuration
    parser.add_argument("--dtype", type=str, default="bfloat16",
                        choices=["float16", "bfloat16", "float32"])
    parser.add_argument("--num_workers", type=int, default=4)
    parser.add_argument("--device", type=str, default="cuda:0")

    # Logging and checkpointing
    parser.add_argument("--log_interval", type=int, default=100)
    parser.add_argument("--save_interval", type=int, default=500)
    parser.add_argument("--resume", action="store_true")

    # WandB
    parser.add_argument("--use_wandb", action="store_true")
    parser.add_argument("--wandb_project", type=str, default="MiniMind-Finetune")

    args = parser.parse_args()

    main(args)

Overwriting finetune_ddp.py


In [ ]:
cd /content/drive/MyDrive/gpt3

/content/drive/MyDrive/gpt3


In [ ]:
!python finetune_ddp.py \
    --data_path data/processed/finetune.jsonl \
    --tokenizer_path model/tokenizer.json \
    --save_dir out_finetune_v2 \
    --pretrained_checkpoint out/checkpoints/checkpoint_latest.pt \
    --hidden_size 768 \
    --num_hidden_layers 16 \
    --num_attention_heads 16 \
    --num_key_value_heads 8 \
    --max_seq_len 512 \
    --dropout 0 \
    --use_moe 0 \
    --epochs 8 \
    --batch_size 16 \
    --learning_rate 1.5e-5 \
    --min_lr 1e-6 \
    --warmup_steps 500 \
    --weight_decay 0.01 \
    --grad_clip 1.0 \
    --accumulation_steps 1 \
    --dtype bfloat16 \
    --num_workers 4 \
    --log_interval 50 \
    --save_interval 300

2025-11-11 21:13:49.949649: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-11 21:13:49.967402: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762895629.989175    1647 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762895629.995803    1647 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762895630.012402    1647 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

# **TEST THE FINE TUNE**

# 8. Test Fine-Tuned Model

In [ ]:
%%writefile test_sft.py
"""
6_inference.py - Test Your Finetuned Model

Simple script to generate responses from your finetuned model.

Usage:
    python 6_inference.py --model out_finetune/final_model
    python 6_inference.py --checkpoint out_finetune/checkpoints/checkpoint_best.pt
"""

import os
import sys
import argparse
import torch
from pathlib import Path

# Add parent directory to path for imports
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

try:
    from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
except ImportError:
    print("❌ Could not import MiniMind model")
    print("Please ensure model/model_minimind.py exists")
    sys.exit(1)

# ============================================================================
# Tokenizer Wrapper
# ============================================================================
class EnglishTokenizer:
    """Simple tokenizer wrapper"""
    def __init__(self, tokenizer_path='model/tokenizer.json'):
        from tokenizers import Tokenizer

        self.tokenizer = Tokenizer.from_file(tokenizer_path)
        self.vocab_size = self.tokenizer.get_vocab_size()

        vocab = self.tokenizer.get_vocab()
        self.pad_id = vocab.get('<pad>', 0)
        self.unk_id = vocab.get('<unk>', 1)
        self.bos_id = vocab.get('<s>', 2)
        self.eos_id = vocab.get('</s>', 3)

    def encode(self, text, add_special_tokens=True):
        encoding = self.tokenizer.encode(text)
        ids = encoding.ids
        if add_special_tokens:
            ids = [self.bos_id] + ids + [self.eos_id]
        return ids

    def decode(self, ids):
        if isinstance(ids, torch.Tensor):
            ids = ids.tolist()
        # Remove special tokens
        ids = [id for id in ids if id not in [self.pad_id, self.bos_id, self.eos_id]]
        return self.tokenizer.decode(ids)

    def __len__(self):
        return self.vocab_size

# ============================================================================
# Model Loading
# ============================================================================

def load_model_from_pretrained(model_dir, device='cuda'):
    """Load model from HuggingFace-style directory"""
    print(f"Loading model from {model_dir}...")

    model = MiniMindForCausalLM.from_pretrained(model_dir)
    model = model.to(device)
    model.eval()

    print(f"✅ Model loaded successfully")
    return model

def load_model_from_checkpoint(checkpoint_path, tokenizer, device='cuda',
                                hidden_size=768, num_hidden_layers=16,
                                num_attention_heads=16, num_key_value_heads=8):
    """Load model from checkpoint file"""
    print(f"Loading model from checkpoint {checkpoint_path}...")

    # Create config
    config = MiniMindConfig(
        hidden_size=hidden_size,
        num_hidden_layers=num_hidden_layers,
        num_attention_heads=num_attention_heads,
        num_key_value_heads=num_key_value_heads,
        vocab_size=len(tokenizer),
        max_position_embeddings=2048,
        bos_token_id=tokenizer.bos_id,
        eos_token_id=tokenizer.eos_id,
    )

    # Create model
    model = MiniMindForCausalLM(config)

    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location='cpu')

    if 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
    else:
        state_dict = checkpoint

    model.load_state_dict(state_dict, strict=False)
    model = model.to(device)
    model.eval()

    print(f"✅ Model loaded successfully")
    return model

# ============================================================================
# Text Generation
# ============================================================================

@torch.no_grad()
def generate_response(model, tokenizer, instruction, device='cuda',
                     max_length=256, temperature=0.7, top_p=0.9,
                     top_k=50, repetition_penalty=1.1):
    """
    Generate response for a given instruction.

    Args:
        model: The language model
        tokenizer: Tokenizer
        instruction: Input instruction text
        device: Device to run on
        max_length: Maximum generation length
        temperature: Sampling temperature (higher = more random)
        top_p: Nucleus sampling threshold
        top_k: Top-k sampling threshold
        repetition_penalty: Penalty for repeating tokens

    Returns:
        Generated response text
    """
    # Encode instruction
    input_ids = tokenizer.encode(instruction, add_special_tokens=True)
    input_ids = torch.tensor([input_ids], dtype=torch.long).to(device)

    # Track generated tokens
    generated_ids = input_ids.clone()

    # Generation loop
    for _ in range(max_length):
        # Forward pass
        outputs = model(generated_ids[:, -512:])  # Use last 512 tokens for context
        logits = outputs.logits[:, -1, :]  # Get logits for last token

        # Apply temperature
        logits = logits / temperature

        # Apply repetition penalty
        if repetition_penalty != 1.0:
            for token_id in set(generated_ids[0].tolist()):
                logits[0, token_id] /= repetition_penalty

        # Apply top-k filtering
        if top_k > 0:
            indices_to_remove = logits[0] < torch.topk(logits[0], top_k)[0][..., -1, None]
            logits[0, indices_to_remove] = float('-inf')

        # Apply top-p (nucleus) filtering
        if top_p < 1.0:
            sorted_logits, sorted_indices = torch.sort(logits[0], descending=True)
            cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)

            # Remove tokens with cumulative probability above the threshold
            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0

            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            logits[0, indices_to_remove] = float('-inf')

        # Sample next token
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs[0], num_samples=1)

        # Check for EOS
        if next_token.item() == tokenizer.eos_id:
            break

        # Append to generated sequence
        generated_ids = torch.cat([generated_ids, next_token.unsqueeze(0)], dim=1)

    # Decode only the generated part (skip instruction)
    response_ids = generated_ids[0, len(input_ids[0]):]
    response = tokenizer.decode(response_ids)

    return response.strip()

# ============================================================================
# Interactive Mode
# ============================================================================

def interactive_mode(model, tokenizer, device='cuda', **gen_kwargs):
    """Interactive chat mode"""
    print("\n" + "="*70)
    print("INTERACTIVE MODE")
    print("="*70)
    print("\nType your instruction and press Enter.")
    print("Type 'quit' or 'exit' to stop.")
    print("Type 'params' to adjust generation parameters.")
    print("="*70 + "\n")

    while True:
        try:
            instruction = input("You: ").strip()

            if not instruction:
                continue

            if instruction.lower() in ['quit', 'exit', 'q']:
                print("Goodbye!")
                break

            if instruction.lower() == 'params':
                print("\nCurrent parameters:")
                for key, value in gen_kwargs.items():
                    print(f"  {key}: {value}")
                print()
                continue

            # Generate response
            print("Assistant: ", end="", flush=True)

            response = generate_response(
                model, tokenizer, instruction, device, **gen_kwargs
            )

            print(response)
            print()

        except KeyboardInterrupt:
            print("\n\nGoodbye!")
            break
        except Exception as e:
            print(f"\n❌ Error: {e}")
            continue

# ============================================================================
# Batch Testing
# ============================================================================

def batch_test(model, tokenizer, test_cases, device='cuda', **gen_kwargs):
    """Test model with multiple instructions"""
    print("\n" + "="*70)
    print("BATCH TESTING")
    print("="*70 + "\n")

    for i, instruction in enumerate(test_cases, 1):
        print(f"Test {i}/{len(test_cases)}")
        print(f"Instruction: {instruction}")
        print(f"Response: ", end="", flush=True)

        response = generate_response(
            model, tokenizer, instruction, device, **gen_kwargs
        )

        print(response)
        print("-" * 70 + "\n")

# ============================================================================
# Main
# ============================================================================

def main():
    parser = argparse.ArgumentParser(description="Test finetuned MiniMind model")

    # Model loading
    parser.add_argument("--model", type=str, default=None,
                       help="Path to model directory (HuggingFace format)")
    parser.add_argument("--checkpoint", type=str, default=None,
                       help="Path to checkpoint file (.pt)")
    parser.add_argument("--tokenizer", type=str, default="model/tokenizer.json",
                       help="Path to tokenizer.json")

    # Model config (only needed if loading from checkpoint)
    parser.add_argument("--hidden_size", type=int, default=768)
    parser.add_argument("--num_hidden_layers", type=int, default=16)
    parser.add_argument("--num_attention_heads", type=int, default=16)
    parser.add_argument("--num_key_value_heads", type=int, default=8)

    # Generation parameters
    parser.add_argument("--max_length", type=int, default=256,
                       help="Maximum generation length")
    parser.add_argument("--temperature", type=float, default=0.7,
                       help="Sampling temperature (higher = more random)")
    parser.add_argument("--top_p", type=float, default=0.9,
                       help="Nucleus sampling threshold")
    parser.add_argument("--top_k", type=int, default=50,
                       help="Top-k sampling threshold")
    parser.add_argument("--repetition_penalty", type=float, default=1.1,
                       help="Repetition penalty")

    # Mode
    parser.add_argument("--mode", type=str, default="interactive",
                       choices=["interactive", "batch"],
                       help="Inference mode")
    parser.add_argument("--test_file", type=str, default=None,
                       help="File with test instructions (one per line)")

    # Device
    parser.add_argument("--device", type=str, default="cuda",
                       help="Device to run on (cuda/cpu)")

    args = parser.parse_args()

    # Check device
    if args.device == 'cuda' and not torch.cuda.is_available():
        print("⚠️  CUDA not available, using CPU")
        args.device = 'cpu'

    # Load tokenizer
    print("Loading tokenizer...")
    tokenizer = EnglishTokenizer(args.tokenizer)
    print(f"✅ Tokenizer loaded (vocab size: {len(tokenizer):,})")

    # Load model
    if args.model:
        model = load_model_from_pretrained(args.model, args.device)
    elif args.checkpoint:
        model = load_model_from_checkpoint(
            args.checkpoint, tokenizer, args.device,
            args.hidden_size, args.num_hidden_layers,
            args.num_attention_heads, args.num_key_value_heads
        )
    else:
        print("❌ Please provide --model or --checkpoint")
        sys.exit(1)

    # Generation kwargs
    gen_kwargs = {
        'max_length': args.max_length,
        'temperature': args.temperature,
        'top_p': args.top_p,
        'top_k': args.top_k,
        'repetition_penalty': args.repetition_penalty,
    }

    # Run inference
    if args.mode == 'interactive':
        interactive_mode(model, tokenizer, args.device, **gen_kwargs)

    elif args.mode == 'batch':
        if args.test_file:
            with open(args.test_file, 'r') as f:
                test_cases = [line.strip() for line in f if line.strip()]
        else:
            # Default test cases
            test_cases = [
                "What is machine learning?",
                "Write a Python function to reverse a string",
                "Explain quantum computing in simple terms",
                "What are the benefits of exercise?",
                "How do neural networks work?",
            ]

        batch_test(model, tokenizer, test_cases, args.device, **gen_kwargs)

if __name__ == "__main__":
    main()

Overwriting test_sft.py


In [ ]:
!python test_sft.py --checkpoint out_finetune/checkpoints/checkpoint_best.pt --checkpoint out_finetune/checkpoints/checkpoint_latest.pt \
--hidden_size 768 --num_hidden_layers 16


2025-11-11 12:12:46.863830: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-11 12:12:46.881590: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762863166.902718    2708 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762863166.909135    2708 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762863166.925442    2708 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## Test Examples

## Interactive Testing

Try your own instructions!

In [ ]:
# Interactive testing
instruction = input("Enter your instruction: ")
print(f"\n🤖 Response: {generate_response(instruction)}\n")

# 9. Download Your Model

## Download Fine-Tuned Model and Tokenizer

# 10. Summary

## 🎉 Congratulations!

You've successfully fine-tuned your Transformer model on 52K instructions!

### What You Accomplished:
- ✅ Downloaded and formatted Alpaca dataset
- ✅ Loaded pretrained model from Chapter 8
- ✅ Fine-tuned on instruction-response pairs
- ✅ Created instruction-following AI
- ✅ Tested and verified model performance

### Your Model Can Now:
- Answer questions
- Follow instructions
- Perform tasks
- Generate creative content

### Next Steps:
1. Download your model files
2. Use `test_sft.py` script locally for more testing
3. Deploy in your applications
4. Try domain-specific fine-tuning

---

### Training Costs:
- Chapter 8 (Pretraining): ~$25
- Chapter 9 (SFT): ~$5-7
- **Total: ~$30-32** for your own instruction-following LLM!

---

## 📚 Documentation:
- Full documentation in Chapter 9
- Test scripts available in the package
- See `CHAPTER_9_COMPLETE_GUIDE.md` for details

**Happy training! 🚀**